Install dependencies and load data

In [ ]:
!pip install transformers datasets scikit-learn torch accelerate tqdm -q

In [ ]:
import os, random, numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.utils.data import WeightedRandomSampler
from transformers import (
    RobertaTokenizerFast, RobertaForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME   = 'roberta-base'
MAX_LEN      = 512
BATCH_SIZE   = 16
EPOCHS       = 20
LR           = 5e-6
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
OUTPUT_DIR   = './roberta_classifier_final'

LABEL2ID = {'not_green_claim': 0, 'green_claim': 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [ ]:
df = pd.read_csv("../../output/labelled/labelled.csv")
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

Split and tokenize labelled sentences into train-val-test

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["Label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["Label"], random_state=SEED
)

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

class TopicDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts, self.labels = df['Sentence'].tolist(), df['Label'].tolist()
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)}
train_dataset = TopicDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = TopicDataset(val_df,   tokenizer, MAX_LEN)
test_dataset  = TopicDataset(test_df,  tokenizer, MAX_LEN)

class_counts   = train_df['Label'].value_counts().sort_index().values
sample_weights = [1.0 / class_counts[label] for label in train_df['Label']]
sampler        = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True) 
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)

Load model and define helpers

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
).to(DEVICE)

optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps  = len(train_loader) * EPOCHS
scheduler    = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * WARMUP_RATIO), total_steps)

class_counts   = train_df['Label'].value_counts().sort_index().values
class_weights  = torch.tensor([1.0 / c for c in class_counts], dtype=torch.float).to(DEVICE)
criterion      = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
def run_epoch(model, loader, optimizer=None, scheduler=None, train=True, threshold=0.5):
    model.train() if train else model.eval()
    total_loss, probs_all, labels_all = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    desc = 'Train' if train else 'Val'

    with ctx:
        for batch in tqdm(loader, desc=desc, leave=False):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['labels'].to(DEVICE)

            if train: optimizer.zero_grad()
            out = model(input_ids=ids, attention_mask=mask, labels=lbls)
            if train:
                out.loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()

            total_loss += out.loss.item()
            probs = torch.softmax(out.logits, dim=-1)[:, 1].detach().cpu().numpy()
            probs_all.extend(probs)
            labels_all.extend(lbls.detach().cpu().numpy())

    # Apply threshold to get final predictions
    preds_all = (np.array(probs_all) > threshold).astype(int)
    f1  = f1_score(labels_all, preds_all, pos_label=LABEL2ID['green_claim'], zero_division=0)
    acc = accuracy_score(labels_all, preds_all)
    return total_loss / len(loader), acc, f1, np.array(probs_all), np.array(labels_all)


def find_best_threshold(probs, labels, pos_label=1):
    """Search over thresholds to maximise val F1."""
    precisions, recalls, thresholds = precision_recall_curve(labels, probs, pos_label=pos_label)
    f1s = np.where(
        (precisions + recalls) == 0, 0,
        2 * precisions * recalls / (precisions + recalls)
    )
    best_idx = np.argmax(f1s[:-1])   # last element has no matching threshold
    return float(thresholds[best_idx]), float(f1s[best_idx])

Training Loop

In [ ]:
best_val_f1, best_threshold = 0.0, 0.5
rounds_without_improve = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_f1, _, _ = run_epoch(
        model, train_loader, optimizer, scheduler, train=True, threshold=0.5
    )
    vl_loss, _, _, val_probs, val_labels = run_epoch(
        model, val_loader, train=False, threshold=0.5
    )
    opt_thresh, vl_f1 = find_best_threshold(val_probs, val_labels, pos_label=LABEL2ID['green_claim'])

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(accuracy_score(val_labels, (val_probs > opt_thresh).astype(int)))

    print(f'Epoch {epoch} — tr_f1 (0.5): {tr_f1:.4f}  |  vl_f1 (thresh={opt_thresh:.2f}): {vl_f1:.4f}')

    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_threshold = opt_thresh
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        np.save(os.path.join(OUTPUT_DIR, 'best_threshold.npy'), np.array([best_threshold]))
        print(f'Saved best model (val_f1={best_val_f1:.4f}, threshold={best_threshold:.2f})')
        rounds_without_improve = 0
    else:
        rounds_without_improve += 1
        print(f'No improvement ({rounds_without_improve}/{2})')
        if (rounds_without_improve >= 2):
          print(f'Early stopping triggered at epoch {epoch}')
          break


print(f'\nBest val F1: {best_val_f1:.4f} at threshold {best_threshold:.2f}')

Evaluate model

In [ ]:
# Evaluate model with highest val f1 on the test set
best_threshold = float(np.load(os.path.join(OUTPUT_DIR, 'best_threshold.npy'))[0])

best_model = RobertaForSequenceClassification.from_pretrained(
    os.path.abspath(OUTPUT_DIR)
).to(DEVICE)
best_model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        out = best_model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
        )
        probs = torch.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
        preds = (probs > best_threshold).astype(int)

        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].numpy())

print(f"Threshold: {best_threshold}\n")
print(classification_report(
    all_labels, all_preds,
    target_names=list(LABEL2ID.keys()),
    zero_division=0
))

In [ ]:
# Visualize train and val loss over epochs
ep = range(1, EPOCHS)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ep, history['train_loss'], marker='o', label='Train')
axes[0].plot(ep, history['val_loss'],   marker='o', label='Val')
axes[0].set_title('Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].plot(ep, history['train_acc'], marker='o', label='Train')
axes[1].plot(ep, history['val_acc'],   marker='o', label='Val')
axes[1].set_title('Accuracy per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('RoBERTa Fine-tuning — Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Print out false positives and negatives in the test set
error_df = pd.DataFrame({
    'text':      test_df['Sentence'].values,
    'true_label': all_labels,
    'pred_label': all_preds,
    'green_prob': all_probs
})

false_positives = error_df[
    (error_df['pred_label'] == 1) & (error_df['true_label'] == 0)
].sort_values('green_prob', ascending=False)

false_negatives = error_df[
    (error_df['pred_label'] == 0) & (error_df['true_label'] == 1)
].sort_values('green_prob', ascending=True)

print(f"False positives: {len(false_positives)}")
print(f"False negatives: {len(false_negatives)}")

print("\n=== FALSE POSITIVES ===")
for _, row in false_positives.iterrows():
    print(f"[prob={row['green_prob']:.2f}] {row['text']}\n")

print("\n=== FALSE NEGATIVES ===")
for _, row in false_negatives.iterrows():
    print(f"[prob={row['green_prob']:.2f}] {row['text']}\n")